# NLP Practical 3 — Word-Level Analysis

Run in **Google Colab**. Cells with `pip install` only need to run once per session.

**Covers:** regex-based information extraction, two-level morphological analysis, noisy-channel spelling correction, and POS tagging via three approaches (rule-based, HMM/Viterbi trained from scratch, pretrained).


In [ ]:
!pip install nltk -q
import nltk
nltk.download("punkt")
nltk.download("averaged_perceptron_tagger")
nltk.download("words")
nltk.download("treebank")
nltk.download("universal_tagset")


In [ ]:
# ============================================================
# PART A: Regular expressions for information extraction
# ============================================================
import re

text = """Contact us at support@nielit.ac.in or admissions@university.edu.
Meeting scheduled on 12/08/2026 and follow-up on 2026-09-01."""

emails = re.findall(r"[\w.\-]+@[\w\-]+\.[a-zA-Z.]+", text)
dates = re.findall(r"\b\d{1,2}/\d{1,2}/\d{4}\b|\b\d{4}-\d{2}-\d{2}\b", text)

print("Emails found:", emails)
print("Dates found:", dates)


In [ ]:
# ============================================================
# PART B: Two-level morphology (lexical <-> surface) for a small paradigm
# ============================================================
lexicon = {
    "play": {"pos": "V"}, "study": {"pos": "V"}, "box": {"pos": "N"}, "city": {"pos": "N"},
}

def two_level_analyze(surface):
    # simple orthographic rules: y->ies, consonant+y -> ied, default +s/+ed
    for stem in lexicon:
        if surface == stem + "s":
            return stem, "+PRESENT+3SG"
        if surface == stem + "es":
            return stem, "+PRESENT+3SG"
        if stem.endswith("y") and surface == stem[:-1] + "ies":
            return stem, "+PLURAL"
        if stem.endswith("y") and surface == stem[:-1] + "ied":
            return stem, "+PAST"
        if surface == stem + "ed":
            return stem, "+PAST"
    return surface, "+UNKNOWN"

for w in ["plays", "studied", "boxes", "cities", "walked"]:
    print(f"{w:10s} -> {two_level_analyze(w)}")


In [ ]:
# ============================================================
# PART C: Noisy-channel spelling correction
# argmax_w  P(w) * P(observed | w)   approximated via edit distance + frequency
# ============================================================
import nltk
from nltk.corpus import words as nltk_words
from collections import Counter

vocab = Counter(w.lower() for w in nltk_words.words())

def edit_distance_1(word):
    letters = "abcdefghijklmnopqrstuvwxyz"
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    deletes = [a + b[1:] for a, b in splits if b]
    transposes = [a + b[1] + b[0] + b[2:] for a, b in splits if len(b) > 1]
    replaces = [a + c + b[1:] for a, b in splits if b for c in letters]
    inserts = [a + c + b for a, b in splits for c in letters]
    return set(deletes + transposes + replaces + inserts)

def correct(word):
    if word in vocab:
        return word
    candidates = edit_distance_1(word) & set(vocab)
    if not candidates:
        return word  # give up -- would search edit distance 2 in a full system
    return max(candidates, key=lambda w: vocab[w])

for typo in ["speling", "recieve", "langauge"]:
    print(f"{typo:10s} -> {correct(typo)}")

print("\nIndustry link: this is the noisy-channel model behind Gboard/SwiftKey")
print("on-device spelling correction, simplified to edit-distance-1.")


In [ ]:
# ============================================================
# PART D: POS tagging -- rule-based -> HMM/Viterbi -> pretrained
# ============================================================
import nltk

sentence = "The old man will study the new NLP book".split()

# --- Rule-based tagger ---
rules = [
    (r".*ing$", "VBG"), (r".*ed$", "VBD"), (r".*es$", "VBZ"),
    (r".*ould$", "MD"), (r".*\'s$", "NN$"), (r".*s$", "NNS"),
    (r"^(the|a|an)$", "DT"), (r".*", "NN"),
]
rule_tagger = nltk.RegexpTagger(rules)
print("Rule-based tags:", rule_tagger.tag(sentence))

# --- HMM/Viterbi tagger trained on the Penn Treebank sample bundled with NLTK ---
nltk.download("treebank")
from nltk.corpus import treebank
train_data = treebank.tagged_sents()[:2500]
hmm_tagger = nltk.HiddenMarkovModelTagger.train(train_data)
print("\nHMM/Viterbi tags:", hmm_tagger.tag(sentence))

# --- Pretrained (industrial-grade) tagger ---
pretrained_tags = nltk.pos_tag(sentence)
print("\nPretrained (averaged perceptron) tags:", pretrained_tags)
